In [0]:
from pyspark.sql import functions as F

CATALOG = "dbw_taxrisk_dev"
dbutils.widgets.text("country_code", "CO")
COUNTRY_CODE = dbutils.widgets.get("country_code")
LANDING = f"/Volumes/{CATALOG}/bronze/landing"

truth = (spark.read.json(f"{LANDING}/ground_truth/labels_batch_001.jsonl")
         .select("post_id", "true_class"))

cal = (spark.table(f"{CATALOG}.silver.posts_classified").join(truth, "post_id")
       .groupBy("pred_class")
       .agg(F.count("*").alias("n_posts"),
            F.round(F.avg((F.col("true_class") == "in_kind").cast("double")), 4).alias("p_in_kind"))
       .withColumn("country_code", F.lit(COUNTRY_CODE))
       .withColumn("source", F.lit("synthetic ground truth, full set, no holdout"))
       .withColumn("calibrated_at", F.current_timestamp()))

(cal.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.config.class_calibration"))
display(cal.orderBy("pred_class"))